7. In `./data/ames_prices.csv`, create an indicator that equals 1 when `Central.Air == 'Y'` and 0 otherwise.
   
   a. Estimate the proportion of homes with central air.
   
   b. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of this proportion. Report the seed and number of bootstrap replications.
   
   c. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).
   
   d. Using the original estimate and the bootstrap standard error, construct Normal 90%, 95%, and 99% confidence intervals.
   
   e. Use each confidence interval to evaluate $H_0:p=.90$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret your results.



In [1]:
import pandas as pd
import numpy as np 
from scipy.stats import norm

df = pd.read_csv('data/ames_prices.csv')
x = (df['Central.Air'] == 'Y').astype(int)
p_hat = x.mean()

seed = 1234
R = 2_000
rng = np.random.default_rng(seed)

boot = np.array([rng.choice(x, len(x), replace=True).mean() for _ in range(R)])

se = boot.std(ddof=1)

for level in [.90, .95, .99]:
    z = norm.ppf(1 - (1 - level) /2)
    lo, hi = p_hat - z * se, p_hat + z * se
    print(level, (lo, hi), 'reject' if not lo <= .90 <= hi else 'fail to reject')

print(p_hat, se)
# Interpretation: We reject p=0.90 at each level. This is becasue 0.90 does not fall within any of the CI's. The data
# gives strong evidence that more than 90% of Ames homes have central air, at rouphly 93%. 



0.9 (np.float64(0.9252975079919188), np.float64(0.9409140961036444)) reject
0.95 (np.float64(0.9238016458119752), np.float64(0.942409958283588)) reject
0.99 (np.float64(0.9208780680353061), np.float64(0.945333536060257)) reject
0.9331058020477816 0.004747105716837866


8. In `./data/ames_prices.csv`, estimate the Pearson correlation between `np.arcsinh(area)` and `np.arcsinh(price)`.
   
   a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the correlation by resampling paired rows. Report the seed and number of bootstrap replications.
   
   b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).
   
   c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.
   
   d. Use each confidence interval to evaluate $H_0:\rho=0$. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis. Briefly interpret the association without making a causal claim.
   
   e. Briefly explain why `area` and `price` must remain paired during resampling.



In [14]:
df = pd.read_csv('data/ames_prices.csv')

x = np.arcsinh(df['area']).to_numpy()
y = np.arcsinh(df['price']).to_numpy()
n = len(x)
mean_x = x.mean()
mean_y = y.mean()

# display(f'Count of X NA: {x.isna().sum()}', f'Count of Y NA: {y.isna().sum()}')

seed = 4312
R = 2_000
rng = np.random.default_rng(seed)


r_hat = np.corrcoef(x, y)[0, 1]
#bootstrap 
boot_r = np.empty(R)
for r in range(R):
    idx = rng.integers(0, n, size = n)
    boot_r[r] = np.corrcoef(x[idx], y[idx])[0,1]



se = boot_r.std(ddof=1)

print(f'Seed: {seed}, Replications: {R}')
print(f'r_hat = {r_hat:.4f}, bootstrap SE = {se:.5f}')

for level in [0.90, 0.95, 0.99]:
    alpha = 1 - level
    z = norm.ppf(1 - alpha / 2)
    lo, hi = r_hat - z * se, r_hat + z * se
    decision = 'Reject H0' if not (lo <= 0 <= hi) else 'Fail to reject H0'
    print(f'{level:.0%} CI: ({lo:.4f}, {hi:.4f})  alpha={alpha:.2f}  -> {decision}')






Seed: 4312, Replications: 2000
r_hat = 0.7232, bootstrap SE = 0.01000
90% CI: (0.7067, 0.7396)  alpha=0.10  -> Reject H0
95% CI: (0.7036, 0.7428)  alpha=0.05  -> Reject H0
99% CI: (0.6974, 0.7489)  alpha=0.01  -> Reject H0


D. None of the intervals contain 0 so we reject each significance level. Larger homes tend to sell for higher prices; thus, the association is positive. 
E. Correlation measures how area and price vary together for each individual house. Resampling them seperately would break the correlation. 

9. In `./data/metabric.csv`, use `Overall Survival (Months)` to estimate the mean recorded time to death or last follow-up. Include both living and deceased patients.
   
   a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.
   
   b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (using `ddof=1` ensures that there's the $1/(N-1)$ degrees of freedom correction).
   
   c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.
   
   d. Use each confidence interval to evaluate $H_0:\mu=120$ months. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.
   
   e. Interpret the estimate and confidence intervals in both months and years. Because living patients' eventual death times are right-censored, explain why these results do not estimate unrestricted mean lifetime.


In [18]:
df = pd.read_csv('data/metabric.csv')

x = df['Overall Survival (Months)'].dropna().to_numpy()  
n = len(x)
mu_hat = x.mean()

seed = 9
R = 2_000
rng = np.random.default_rng(seed)
boot = np.array([rng.choice(x, n, replace=True).mean() for _ in range(R)])
se = boot.std(ddof=1)

for level in [0.90, 0.95, 0.99]:
    alpha = 1 - level
    z = norm.ppf(1 - alpha / 2)
    lo, hi = mu_hat - z * se, mu_hat + z * se
    decision = 'Reject H0' if not (lo <= 120 <= hi) else 'Fail to reject H0'
    print(f'{level:.0%} CI: ({lo:.2f}, {hi:.2f}) months = ({lo/12:.2f}, {hi/12:.2f}) years  alpha={alpha:.2f} -> {decision}')
    
print(mu_hat)

90% CI: (125.54, 132.65) months = (10.46, 11.05) years  alpha=0.10 -> Reject H0
95% CI: (124.86, 133.33) months = (10.40, 11.11) years  alpha=0.05 -> Reject H0
99% CI: (123.53, 134.66) months = (10.29, 11.22) years  alpha=0.01 -> Reject H0
129.09429138744105


D. 120 is below all three intervals, so reject the null hypothesis at .10, .05 and .01.
E. The mean recorded time of death or last follow-up is 129.1 months or 10.8 years. Patients still alive, the recorded time is when the follow-up ended, not when they died. Thus, their true lifetimes are longer than recorded. This makes the mean an underestimate. 95% confidence interval [10.4, 11.1] years. 

10. In `./data/nhanes_data_17_18.csv`, select respondents who are at least 18 years old and have an observed value for `SleepHoursWeekdaysOrWorkdays`. Estimate their mean weekday sleep.
   
   a. Using a reproducible random-number seed, generate at least 2,000 bootstrap estimates of the mean. Report the seed and number of bootstrap replications.
   
   b. Compute the bootstrap standard error as the sample standard deviation of the bootstrap estimates (`ddof=1`).
   
   c. Using the original estimate and the bootstrap standard error, construct centered-Normal 90%, 95%, and 99% confidence intervals.
   
   d. Use each confidence interval to evaluate $H_0:\mu=8$ hours. At the matching significance levels $\alpha=.10,.05,.01$, state whether you reject or fail to reject the null hypothesis.
   
   e. Interpret the result and discuss whether its statistical and practical significance tell the same story. For this exercise, treat observations as independent and unweighted. Because the complex survey design is ignored, do not describe the result as nationally representative or the intervals as survey-adjusted.

In [24]:
df = pd.read_csv('./data/nhanes_data_17_18.csv')
adults = df[(df['AgeInYearsAtScreening'] >= 18) & df['SleepHoursWeekdaysOrWorkdays'].notna()]
x = adults['SleepHoursWeekdaysOrWorkdays'].to_numpy()
n = len(x)
mu_hat = x.mean()

seed = 10
R = 2_000
rng = np.random.default_rng(seed)
boot = np.array([rng.choice(x, n, replace=True).mean() for _ in range(R)])
se = boot.std(ddof=1)

print(f'Seed: {seed}, Replications: {R}, n = {n}')
print(f'mean = {mu_hat:.3f} hours, SE = {se:.4f}')

for level in [0.90, 0.95, 0.99]:
    alpha = 1 - level
    z = norm.ppf(1 - alpha / 2)
    lo, hi = mu_hat - z * se, mu_hat + z * se
    decision = 'Reject H0' if not (lo <= 8 <= hi) else 'Fail to reject H0'
    print(f'{level:.0%} CI: ({lo:.3f}, {hi:.3f})  alpha={alpha:.2f} -> {decision}')

/var/folders/_d/chnmr63s10311_dffvmdtd140000gn/T/ipykernel_86033/4057363257.py:1: DtypeWarning: Columns (142: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/nhanes_data_17_18.csv')


Seed: 10, Replications: 2000, n = 5489
mean = 7.621 hours, SE = 0.0225
90% CI: (7.584, 7.658)  alpha=0.10 -> Reject H0
95% CI: (7.577, 7.665)  alpha=0.05 -> Reject H0
99% CI: (7.563, 7.679)  alpha=0.01 -> Reject H0


D. 8 is outside all 3 so we reject the null hypothesis.
E. Each of the intervals are below 8. The large n makes SE tiny. The gap is about 23 minutes a night so it is small but statistically significant. The results teat respondents as independent and unweighted so they describe this sample only. They are not nationally representative, and the intervals are not survey adjusted. 